# Notebook 2: The REPL Sandbox

In this notebook, you'll learn:
- Why RLMs need code execution (the model writes Python to manipulate context)
- How to build a safe sandbox for executing LLM-generated code
- How to capture output, inject variables, and handle errors
- Safety: blocking dangerous operations like file system access

## The Key Insight
Traditional LLMs see the entire context in their prompt window. RLMs take a different approach: the context exists as a **variable** in a Python environment, and the model writes **code** to examine and process it.

## Step 1: Why Code Execution?

Imagine you have a 100,000 word document and need to find a specific fact. Two approaches:

1. **Traditional**: Stuff the entire document into the LLM's prompt. Hope it can process it all.
2. **RLM**: Store the document as a variable. Let the LLM write code like `re.search("secret code", context)` to find what it needs.

The RLM approach is more efficient — the model only processes what it needs. But it requires a way to **safely execute code** the model generates.

## Step 2: Raw exec() — The Dangerous Way

Python has a built-in `exec()` function that runs code from a string. Let's see it in action:

In [ ]:
# Python can execute code from strings!
exec("print(2 + 2)")
exec("x = 42")
exec("print(f'x is {x}')")

## The Problem with Raw exec()

This works, but it's **dangerous**:
- The code could access the file system (`import os; os.remove('important_file')`)
- It could make network requests
- It could run system commands
- There's no way to capture the output programmatically
- Errors crash our entire program

We need a **sandbox** — a restricted environment where code runs safely.

## Step 3: Our Sandbox Class

Let's use the `Sandbox` class from our `rlm_core` library:

In [ ]:
import sys
sys.path.insert(0, "..")

from rlm_core import Sandbox

# Create a sandbox and run some code
sb = Sandbox()
result = sb.execute("print('Hello from the sandbox!')")

print("Captured stdout:", repr(result.stdout))
print("Error:", result.error)

## Step 4: Capturing Output

The sandbox captures everything printed to stdout. This is how we see what the LLM's code produces:

In [ ]:
sb = Sandbox()

# The sandbox captures all print output
result = sb.execute("""
for i in range(5):
    print(f"Item {i}: {'even' if i % 2 == 0 else 'odd'}")
""")

print("=== Captured Output ===")
print(result.stdout)

## Step 5: Variable Persistence

Variables created in one `execute()` call persist for the next. This is crucial — the LLM's code builds up state across multiple execution rounds:

In [ ]:
sb = Sandbox()

# First execution: create a variable
sb.execute("total = 0")

# Second execution: use and modify it
sb.execute("total += 10")
sb.execute("total += 20")

# Third execution: read it
result = sb.execute("print(f'Total is: {total}')")
print(result.stdout)

# We can also read variables directly
print("Direct access:", sb.get_variable("total"))

## Step 6: Variable Injection — The Bridge to Context

This is where the magic happens. We can **inject variables** into the sandbox before the LLM's code runs. This is how the RLM provides the `context` variable:

In [ ]:
# Simulate an RLM scenario: inject a document as "context"
document = """
The Great Wall of China stretches over 13,000 miles across northern China.
Construction began in the 7th century BC and continued for centuries.
The wall was built to protect against invasions from northern nomadic groups.
"""

sb = Sandbox(variables={"context": document})

# Now the LLM's code can access it
result = sb.execute("""
lines = context.strip().split('\\n')
print(f"Document has {len(lines)} lines")
for i, line in enumerate(lines):
    if line.strip():
        print(f"  Line {i}: {line.strip()[:50]}...")
""")
print(result.stdout)

## Step 7: Error Handling — When the Model Writes Bad Code

LLMs don't always write perfect code. The sandbox catches errors gracefully instead of crashing:

In [ ]:
sb = Sandbox()

# Runtime error
result = sb.execute("x = 1 / 0")
print("=== Runtime Error ===")
print("Error:", result.error[:200])  # Truncate for readability

print()

# Syntax error
result = sb.execute("def foo(")
print("=== Syntax Error ===")
print("Error:", result.error[:200])

print()

# Undefined variable
result = sb.execute("print(undefined_var)")
print("=== Name Error ===")
print("Error:", result.error[:200])

## Step 8: Safety Restrictions

Our sandbox blocks dangerous operations. The LLM cannot access the file system, network, or system commands:

In [ ]:
sb = Sandbox()

# Try to import os — BLOCKED!
result = sb.execute("import os")
print("import os:", "BLOCKED!" if result.error else "allowed")

# Try subprocess — BLOCKED!
result = sb.execute("import subprocess")
print("import subprocess:", "BLOCKED!" if result.error else "allowed")

# Try socket — BLOCKED!
result = sb.execute("import socket")
print("import socket:", "BLOCKED!" if result.error else "allowed")

# Safe imports still work
result = sb.execute("import re; print(re.findall(r'\\d+', 'abc 123 def 456'))")
print("\nimport re (safe):", result.stdout.strip())

## Step 9: Injecting Functions — The llm_query() Preview

We can inject **functions** into the sandbox too. This is how the RLM provides `llm_query()` — a function that lets the model call itself recursively:

In [ ]:
# Simulate what llm_query() will look like
def fake_llm_query(question):
    """Pretend to be an LLM answering questions."""
    return f"[LLM would answer: {question}]"

sb = Sandbox(variables={
    "context": "The capital of France is Paris.",
    "llm_query": fake_llm_query,
})

result = sb.execute("""
# The model's code can call llm_query() to ask sub-questions
answer = llm_query("What is the capital mentioned in the context?")
print(f"Sub-query result: {answer}")

# It can also directly process the context
import re
if re.search(r'capital.*Paris', context):
    print("Found: Paris is the capital!")
""")
print(result.stdout)

## Exercise: Build a Mini REPL

Try writing your own code strings and executing them in the sandbox. Modify the code below:

In [ ]:
# Exercise: Try your own code in the sandbox!
sb = Sandbox(variables={"context": "The secret code is BLUE-FALCON-42. Don't tell anyone!"})

# Write code that would find the secret code using regex
your_code = """
import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
if match:
    print(f"Found the secret: {match.group(1)}")
else:
    print("No secret found")
"""

result = sb.execute(your_code)
print(result.stdout)

## Key Takeaways

1. **RLMs need code execution** — the model writes Python to manipulate context stored as a variable
2. **The Sandbox** safely executes code with output capture and error handling
3. **Variable injection** bridges the gap between our data and the model's code
4. **Function injection** enables recursive calls (next notebook!)
5. **Safety restrictions** prevent the model from accessing dangerous system operations

## What's Next?

In Notebook 3, we'll connect the LLM client to the sandbox — the model will write code, the sandbox executes it, and `llm_query()` lets the model **call itself recursively**. This is the heart of the RLM framework.